In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!unzip AnswerSheet.zip

Archive:  AnswerSheet.zip
   creating: AnswerSheet/
   creating: AnswerSheet/exam0/
  inflating: AnswerSheet/exam0/exam0_10_1.png  
  inflating: AnswerSheet/exam0/exam0_10_2.png  
  inflating: AnswerSheet/exam0/exam0_10_3.png  
  inflating: AnswerSheet/exam0/exam0_11_1.png  
  inflating: AnswerSheet/exam0/exam0_11_2.png  
  inflating: AnswerSheet/exam0/exam0_11_3.png  
  inflating: AnswerSheet/exam0/exam0_12_1.png  
  inflating: AnswerSheet/exam0/exam0_12_2.png  
  inflating: AnswerSheet/exam0/exam0_12_3.png  
  inflating: AnswerSheet/exam0/exam0_13_1.png  
  inflating: AnswerSheet/exam0/exam0_13_2.png  
  inflating: AnswerSheet/exam0/exam0_13_3.png  
  inflating: AnswerSheet/exam0/exam0_14_1.png  
  inflating: AnswerSheet/exam0/exam0_14_2.png  
  inflating: AnswerSheet/exam0/exam0_14_3.png  
  inflating: AnswerSheet/exam0/exam0_15_1.png  
  inflating: AnswerSheet/exam0/exam0_15_2.png  
  inflating: AnswerSheet/exam0/exam0_15_3.png  
  inflating: AnswerSheet/exam0/exam0_16_1.png  
  in

In [7]:
import os
import cv2
import pandas as pd
import numpy as np
import scipy.io as sio
from sklearn.model_selection import StratifiedKFold, train_test_split
import shutil
from tqdm import tqdm

# ==========================================
# 1. CẤU HÌNH ĐƯỜNG DẪN
# ==========================================
MAT_FILE_PATH = "/content/exams.mat"
BASE_IMG_DIR = "/content/AnswerSheet"

# Nơi lưu Master Pool (Tạm thời)
MASTER_SHEETS_DIR = "Master_Dataset/Sheets"
MASTER_ROIS_DIR = "Master_Dataset/ROIs"

# TÊN 2 THƯ MỤC GỐC MỚI THEO YÊU CẦU
K_FOLDS_SHEETS_DIR = "/content/drive/MyDrive/OMR-Datasets/OMR_5Fold_Sheets"
K_FOLDS_ROIS_DIR = "/content/drive/MyDrive/OMR-Datasets/OMR_5Fold_ROIs"

LABEL_MAP = {1: 'confirmed', 2: 'crossedout', 3: 'empty'}

# ==========================================
# 2. ĐỌC VÀ CHUYỂN ĐỔI FILE .MAT
# ==========================================
print("Đang phân tích file metadata exams.mat...")
mat_data = sio.loadmat(MAT_FILE_PATH, squeeze_me=True, struct_as_record=False)
struct_keys = [k for k in mat_data.keys() if not k.startswith('__')]
exam_structs = mat_data[struct_keys[0]]

records = []
for exam in exam_structs:
    img_name = str(exam.imageName)
    exam_id = img_name.split('_')[0]
    ans_types = np.array(exam.answerType).flatten()
    rects_reshaped = np.array(exam.questionRect).reshape(-1, 4)

    for i, (ans, rect) in enumerate(zip(ans_types, rects_reshaped)):
        x, y, w, h = rect
        records.append({
            'image_name': img_name, 'exam_id': exam_id, 'box_idx': i,
            'label': int(ans), 'x': int(x), 'y': int(y), 'w': int(w), 'h': int(h)
        })

df = pd.DataFrame(records)

def get_sheet_id(img_name):
    parts = img_name.replace('.png', '').replace('.jpg', '').split('_')
    return f"{parts[0]}_{parts[1]}"
df['sheet_id'] = df['image_name'].apply(get_sheet_id)

# ==========================================
# 3. TẠO MASTER DATASET (CẮT ẢNH 1 LẦN DUY NHẤT)
# ==========================================
if not os.path.exists(MASTER_ROIS_DIR):
    print("\n--- BƯỚC 1: TẠO MASTER POOL (Đang cắt ảnh...) ---")
    for exam_idx in range(6):
        os.makedirs(os.path.join(MASTER_SHEETS_DIR, f"exam{exam_idx}"), exist_ok=True)
    for cls in LABEL_MAP.values():
        os.makedirs(os.path.join(MASTER_ROIS_DIR, cls), exist_ok=True)

    grouped = df.groupby('image_name')
    for img_name, bubbles in tqdm(grouped, desc="Processing Images"):
        exam_folder = bubbles['exam_id'].iloc[0]
        img_name_ext = img_name if img_name.endswith(('.png', '.jpg')) else img_name + '.png'
        src_img_path = os.path.join(BASE_IMG_DIR, exam_folder, img_name_ext)
        if not os.path.exists(src_img_path): continue

        shutil.copy2(src_img_path, os.path.join(MASTER_SHEETS_DIR, exam_folder, img_name_ext))

        img = cv2.imread(src_img_path)
        if img is None: continue

        for _, row in bubbles.iterrows():
            x, y, w, h = row['x'], row['y'], row['w'], row['h']
            label_str = LABEL_MAP[row['label']]
            roi = img[y:y+h, x:x+w]
            if roi.size > 0:
                roi_filename = f"{img_name.replace('.png','')}_box{row['box_idx']}.jpg"
                cv2.imwrite(os.path.join(MASTER_ROIS_DIR, label_str, roi_filename), roi)
else:
    print("\n--- BƯỚC 1: BỎ QUA DO MASTER POOL ĐÃ TỒN TẠI ---")

# ==========================================
# 4. CHIA 5-FOLD (60% TRAIN - 20% VAL - 20% TEST)
# ==========================================
print("\n--- BƯỚC 2: PHÂN BỔ DỮ LIỆU VÀO 2 TẬP FOLDS RIÊNG BIỆT ---")
sheet_stats = df.groupby('sheet_id').agg(
    has_crossed_out=('label', lambda x: 2 in x.values)
).reset_index()
sheet_stats['strat_key'] = sheet_stats['has_crossed_out'].astype(int)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_val_idx, test_idx) in enumerate(skf.split(sheet_stats['sheet_id'], sheet_stats['strat_key'])):
    fold_num = fold + 1
    print(f"Đang sao chép file cho Fold {fold_num}...")

    train_val_sheets = sheet_stats.iloc[train_val_idx]
    train_df, val_df = train_test_split(
        train_val_sheets,
        test_size=0.25,
        random_state=42,
        stratify=train_val_sheets['strat_key']
    )

    train_sheet_ids = train_df['sheet_id'].values
    val_sheet_ids = val_df['sheet_id'].values
    test_sheet_ids = sheet_stats.iloc[test_idx]['sheet_id'].values

    # Tạo cây thư mục tách biệt cho Sheets và ROIs
    for split in ['train', 'val', 'test']:
        for exam_idx in range(6):
            os.makedirs(f"{K_FOLDS_SHEETS_DIR}/Fold_{fold_num}/{split}/exam{exam_idx}", exist_ok=True)
        for cls in LABEL_MAP.values():
            os.makedirs(f"{K_FOLDS_ROIS_DIR}/Fold_{fold_num}/{split}/{cls}", exist_ok=True)

    def distribute_files(sheet_ids, split_name):
        df_split = df[df['sheet_id'].isin(sheet_ids)]

        # 1. Chép vào tập Sheets
        unique_sheets = df_split.drop_duplicates(subset=['image_name'])
        for _, row in unique_sheets.iterrows():
            img_name_ext = row['image_name'] if row['image_name'].endswith(('.png', '.jpg')) else row['image_name'] + '.png'
            exam_folder = row['exam_id']
            src = f"{MASTER_SHEETS_DIR}/{exam_folder}/{img_name_ext}"
            dst = f"{K_FOLDS_SHEETS_DIR}/Fold_{fold_num}/{split_name}/{exam_folder}/{img_name_ext}"
            if os.path.exists(src): shutil.copy2(src, dst)

        # 2. Chép vào tập ROIs
        for _, row in df_split.iterrows():
            label_str = LABEL_MAP[row['label']]
            roi_filename = f"{row['image_name'].replace('.png','')}_box{row['box_idx']}.jpg"
            src = f"{MASTER_ROIS_DIR}/{label_str}/{roi_filename}"
            dst = f"{K_FOLDS_ROIS_DIR}/Fold_{fold_num}/{split_name}/{label_str}/{roi_filename}"
            if os.path.exists(src): shutil.copy2(src, dst)

    distribute_files(train_sheet_ids, 'train')
    distribute_files(val_sheet_ids, 'val')
    distribute_files(test_sheet_ids, 'test')

# ==========================================
# 5. BÁO CÁO TỔNG KẾT
# ==========================================
print("\n==================================================")
print(" 📊 BÁO CÁO THỐNG KÊ 5-FOLD (TÁCH RIÊNG SHEETS & ROIS)")
print("==================================================")

for fold in range(1, 6):
    train_rois = sum(len(files) for _, _, files in os.walk(f"{K_FOLDS_ROIS_DIR}/Fold_{fold}/train"))
    val_rois = sum(len(files) for _, _, files in os.walk(f"{K_FOLDS_ROIS_DIR}/Fold_{fold}/val"))
    test_rois = sum(len(files) for _, _, files in os.walk(f"{K_FOLDS_ROIS_DIR}/Fold_{fold}/test"))

    test_crossedout = len(os.listdir(f"{K_FOLDS_ROIS_DIR}/Fold_{fold}/test/crossedout"))

    print(f"📂 FOLD {fold}:")
    print(f"  - Tập ROIs   : Train={train_rois} | Val={val_rois} | Test={test_rois}")
    print(f"  - ✅ Ảnh Crossed-out trong TEST: {test_crossedout} ô")
    print("-" * 50)

Đang phân tích file metadata exams.mat...

--- BƯỚC 1: BỎ QUA DO MASTER POOL ĐÃ TỒN TẠI ---

--- BƯỚC 2: PHÂN BỔ DỮ LIỆU VÀO 2 TẬP FOLDS RIÊNG BIỆT ---
Đang sao chép file cho Fold 1...


KeyboardInterrupt: 

In [4]:
import os

def print_directory_tree(startpath, max_depth=3):
    """
    Hàm in cây thư mục và đếm số lượng file bên trong mỗi thư mục.
    """
    print(f"📦 CÂY THƯ MỤC: {startpath}")
    print("="*50)

    for root, dirs, files in os.walk(startpath):
        # Tính toán độ sâu của thư mục hiện tại
        level = root.replace(startpath, '').count(os.sep)

        # Giới hạn độ sâu hiển thị
        if level > max_depth:
            # Xóa danh sách dirs để os.walk không đi sâu thêm nữa
            del dirs[:]
            continue

        # Trình bày thụt lề
        indent = '│   ' * level
        is_last_level = (level == max_depth)

        # Tên thư mục hiện tại
        folder_name = os.path.basename(root)
        if folder_name == '':
            folder_name = startpath

        # Đếm số file trực tiếp trong thư mục này (nếu có)
        file_count_str = f" ({len(files)} files)" if files else ""

        if level == 0:
            print(f"📁 {folder_name}")
        else:
            prefix = "└── " if not dirs else "├── "
            print(f"{indent[:-4]}{prefix}📁 {folder_name}{file_count_str}")

# In cây thư mục cho tập Sheets (Giới hạn depth=3 để xem đến thư mục exam)
print_directory_tree("OMR_5Fold_Sheets", max_depth=3)

print("\n\n")

# In cây thư mục cho tập ROIs (Giới hạn depth=3 để xem đến các lớp confirmed/crossedout/empty)
print_directory_tree("OMR_5Fold_ROIs", max_depth=3)

📦 CÂY THƯ MỤC: OMR_5Fold_Sheets
📁 OMR_5Fold_Sheets
├── 📁 Fold_4
│   ├── 📁 test
│   │   └── 📁 exam3 (10 files)
│   │   └── 📁 exam0 (39 files)
│   │   └── 📁 exam4 (12 files)
│   │   └── 📁 exam2 (22 files)
│   │   └── 📁 exam5 (73 files)
│   │   └── 📁 exam1 (17 files)
│   ├── 📁 train
│   │   └── 📁 exam3 (38 files)
│   │   └── 📁 exam0 (60 files)
│   │   └── 📁 exam4 (39 files)
│   │   └── 📁 exam2 (63 files)
│   │   └── 📁 exam5 (220 files)
│   │   └── 📁 exam1 (61 files)
│   ├── 📁 val
│   │   └── 📁 exam3 (12 files)
│   │   └── 📁 exam0 (21 files)
│   │   └── 📁 exam4 (11 files)
│   │   └── 📁 exam2 (18 files)
│   │   └── 📁 exam5 (78 files)
│   │   └── 📁 exam1 (21 files)
├── 📁 Fold_1
│   ├── 📁 test
│   │   └── 📁 exam3 (14 files)
│   │   └── 📁 exam0 (42 files)
│   │   └── 📁 exam4 (15 files)
│   │   └── 📁 exam2 (20 files)
│   │   └── 📁 exam5 (62 files)
│   │   └── 📁 exam1 (22 files)
│   ├── 📁 train
│   │   └── 📁 exam3 (33 files)
│   │   └── 📁 exam0 (57 files)
│   │   └── 📁 exam4 (38 files)
│   │   └

In [8]:
import os
import shutil

# ==========================================
# CẤU HÌNH ĐƯỜNG DẪN
# ==========================================
# Đây là thư mục gốc chứa 5 Fold Sheets mà bạn vừa tạo xong
SRC_SHEETS_DIR = "/content/OMR_5Fold_Sheets"

# Đây là thư mục mới chỉ chứa tập Test
DST_TEST_ONLY_DIR = "/content/OMR_5Fold_Sheets_TestOnly"

print(f"🚀 Đang trích xuất riêng tập Test sang: {DST_TEST_ONLY_DIR}...\n")

# Nếu thư mục đích đã tồn tại thì xóa đi để làm lại cho sạch
if os.path.exists(DST_TEST_ONLY_DIR):
    shutil.rmtree(DST_TEST_ONLY_DIR)

os.makedirs(DST_TEST_ONLY_DIR)

# ==========================================
# COPY TẬP TEST CỦA 5 FOLD
# ==========================================
for fold in range(1, 6):
    fold_name = f"Fold_{fold}"
    src_test_dir = os.path.join(SRC_SHEETS_DIR, fold_name, "test")
    dst_test_dir = os.path.join(DST_TEST_ONLY_DIR, fold_name, "test")

    # Kiểm tra xem thư mục test có tồn tại không rồi copy toàn bộ
    if os.path.exists(src_test_dir):
        # copytree sẽ copy toàn bộ thư mục, bao gồm cả các thư mục con exam0 -> exam5
        shutil.copytree(src_test_dir, dst_test_dir)

        # Đếm số lượng ảnh vừa copy để báo cáo
        num_files = sum(len(files) for _, _, files in os.walk(dst_test_dir))
        print(f"✅ Đã copy thành công {fold_name}/test ({num_files} tờ phiếu)")
    else:
        print(f"❌ Cảnh báo: Không tìm thấy {src_test_dir}")

print("\n🎉 TRÍCH XUẤT HOÀN TẤT!")



🚀 Đang trích xuất riêng tập Test sang: /content/OMR_5Fold_Sheets_TestOnly...

✅ Đã copy thành công Fold_1/test (175 tờ phiếu)
✅ Đã copy thành công Fold_2/test (157 tờ phiếu)
✅ Đã copy thành công Fold_3/test (155 tờ phiếu)
✅ Đã copy thành công Fold_4/test (173 tờ phiếu)
✅ Đã copy thành công Fold_5/test (155 tờ phiếu)

🎉 TRÍCH XUẤT HOÀN TẤT!


In [9]:
# ==========================================
# (TÙY CHỌN) NÉN LẠI ĐỂ LƯU LÊN GOOGLE DRIVE
# ==========================================
# Nếu bạn muốn lưu thành quả này lên Drive để mai dùng, bỏ comment 2 dòng dưới:
!zip -q -r /content/drive/MyDrive/OMR-Datasets/OMR_5Fold_Sheets_TestOnly.zip /content/OMR_5Fold_Sheets_TestOnly
print("📦 Đã nén và lưu file zip lên Google Drive!")

📦 Đã nén và lưu file zip lên Google Drive!


In [5]:
!zip -q -r /content/drive/MyDrive/OMR-Datasets/OMR_5Fold_Sheets.zip OMR_5Fold_Sheets
!zip -q -r /content/drive/MyDrive/OMR-Datasets/OMR_5Fold_ROIs.zip OMR_5Fold_ROIs

Streaming output truncated to the last 5000 lines.
  adding: OMR_5Fold_ROIs/Fold_2/val/confirmed/exam5_50_1_box9.jpg (deflated 4%)
  adding: OMR_5Fold_ROIs/Fold_2/val/confirmed/exam2_67_1_box24.jpg (deflated 3%)
  adding: OMR_5Fold_ROIs/Fold_2/val/confirmed/exam4_52_1_box6.jpg (deflated 6%)
  adding: OMR_5Fold_ROIs/Fold_2/val/confirmed/exam5_83_1_box26.jpg (deflated 5%)
  adding: OMR_5Fold_ROIs/Fold_2/val/confirmed/exam5_222_1_box15.jpg (deflated 5%)
  adding: OMR_5Fold_ROIs/Fold_2/val/confirmed/exam5_47_1_box0.jpg (deflated 4%)
  adding: OMR_5Fold_ROIs/Fold_2/val/confirmed/exam5_24_1_box40.jpg (deflated 18%)
  adding: OMR_5Fold_ROIs/Fold_2/val/confirmed/exam5_128_1_box29.jpg (deflated 5%)
  adding: OMR_5Fold_ROIs/Fold_2/val/confirmed/exam5_88_1_box36.jpg (deflated 6%)
  adding: OMR_5Fold_ROIs/Fold_2/val/confirmed/exam5_190_1_box47.jpg (deflated 4%)
  adding: OMR_5Fold_ROIs/Fold_2/val/confirmed/exam5_57_1_box19.jpg (deflated 4%)
  adding: OMR_5Fold_ROIs/Fold_2/val/confirmed/exam5_251_1

In [10]:
!zip -q -r OMR_5Fold_Sheets_TestOnly.zip /content/OMR_5Fold_Sheets_TestOnly